# 3D Gaussian Splatting v4 — DJI Avata 360 (Final)

**Fixed in v4:** Images right-side-up, OPENCV camera model, 432/432 registered (100%), 240k points.

| Version | Issue | Result |
|---------|-------|--------|
| v1-v2 | Sky-facing (pitch inverted) | COLMAP confused |
| v3 | Right content but upside-down | Blurry splats |
| **v4** | **Correct orientation + OPENCV model** | **432/432 images, 240k pts** |

**Upload to Drive:** `DroneCV/gaussian_splat_data/images_v4.zip` + `colmap_v4_output.zip`

**Runtime:** ~20 min training on A100

In [ ]:
import torch, os, subprocess, shutil, zipfile, glob, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

assert torch.cuda.is_available(), 'GPU required!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/DroneCV/gaussian_splat_data'
DATA_DIR = '/content/data'
MODEL_PATH = '/content/output'

print('\n\u2705 Ready')

In [ ]:
# Prepare data — v4 images (right-side-up) + COLMAP (OPENCV model, 432/432)
os.makedirs(f'{DATA_DIR}/sparse/0', exist_ok=True)
os.makedirs(f'{DATA_DIR}/images', exist_ok=True)

print('Extracting v4 images (ground-facing, correct orientation)...')
with zipfile.ZipFile(f'{DRIVE}/images_v4.zip', 'r') as z:
    for m in tqdm(z.namelist(), desc='Images'):
        z.extract(m, DATA_DIR)

print('\nExtracting COLMAP v4 (OPENCV model, 240k points)...')
with zipfile.ZipFile(f'{DRIVE}/colmap_v4_output.zip', 'r') as z:
    z.extractall('/content/colmap_tmp')

# Move to expected structure
for f in glob.glob('/content/colmap_tmp/colmap_v4/sparse/0/*'):
    shutil.copy(f, f'{DATA_DIR}/sparse/0/')

n_imgs = len(glob.glob(f'{DATA_DIR}/images/*.jpg'))
colmap_files = os.listdir(f'{DATA_DIR}/sparse/0/')
print(f'\n\u2705 {n_imgs} images + COLMAP ({colmap_files})')

# Show samples
import cv2
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
imgs = sorted(glob.glob(f'{DATA_DIR}/images/*.jpg'))
for i in range(6):
    axes[0,i].imshow(cv2.cvtColor(cv2.imread(imgs[i]), cv2.COLOR_BGR2RGB))
    axes[0,i].axis('off'); axes[0,i].set_title(f'View {i+1}')
for i in range(6):
    axes[1,i].imshow(cv2.cvtColor(cv2.imread(imgs[200+i]), cv2.COLOR_BGR2RGB))
    axes[1,i].axis('off'); axes[1,i].set_title(f'View {200+i}')
plt.suptitle(f'{n_imgs} Ground-Facing Images — Correct Orientation', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Install gaussian-splatting
%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive 2>&1 | tail -2
%cd /content/gaussian-splatting
!pip install -q plyfile tqdm
!pip install -q submodules/diff-gaussian-rasterization 2>&1 | tail -1
!pip install -q submodules/simple-knn 2>&1 | tail -1
!pip install -q submodules/fused-ssim 2>&1 | tail -1
print('\n\u2705 Gaussian Splatting installed')

In [ ]:
# Train — 30,000 iterations, production quality
%cd /content/gaussian-splatting

t0 = time.time()

!python train.py \
  -s /content/data \
  --model_path /content/output \
  --iterations 30000 \
  --sh_degree 3 \
  --densify_until_iter 15000 \
  --densify_grad_threshold 0.0002 \
  --opacity_reset_interval 3000 \
  --position_lr_init 0.00016 \
  --position_lr_final 0.0000016 \
  --scaling_lr 0.005 \
  --save_iterations 7000 15000 30000 \
  --test_iterations 7000 15000 30000

elapsed = time.time() - t0
print(f'\n\u2705 Training complete in {elapsed/60:.1f} minutes')

# Verify
if os.path.exists(f'{MODEL_PATH}/cfg_args'):
    ply_files = sorted(glob.glob(f'{MODEL_PATH}/point_cloud/*/point_cloud.ply'))
    for p in ply_files:
        print(f'  {os.path.basename(os.path.dirname(p))}: {os.path.getsize(p)/1e6:.1f} MB')
else:
    print('\u26a0\ufe0f Check output:')
    !find /content/output -type f | head -10

In [ ]:
# Render training views at all checkpoints
%cd /content/gaussian-splatting

for iteration in [7000, 15000, 30000]:
    ply_path = f'{MODEL_PATH}/point_cloud/iteration_{iteration}/point_cloud.ply'
    if os.path.exists(ply_path):
        print(f'Rendering iteration {iteration}...')
        !python render.py -s /content/data --model_path /content/output --iteration {iteration} --skip_test 2>&1 | tail -1

print('\n\u2705 Rendering complete')

In [ ]:
# Progression: 7k vs 15k vs 30k
fig, axes = plt.subplots(3, 6, figsize=(20, 10))

for row, iteration in enumerate([7000, 15000, 30000]):
    render_dir = f'{MODEL_PATH}/train/ours_{iteration}/renders/'
    renders = sorted(glob.glob(f'{render_dir}*.png'))[:6] if os.path.exists(render_dir) else []
    for col in range(6):
        if col < len(renders):
            axes[row, col].imshow(plt.imread(renders[col]))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{iteration} iter', fontsize=12, rotation=0, labelpad=60, va='center')

plt.suptitle('Gaussian Splatting Progression: 7k \u2192 15k \u2192 30k iterations (v4, correct orientation)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/gs_progression_v4.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ground Truth vs Render comparison at 30k
gt_dir = f'{MODEL_PATH}/train/ours_30000/gt/'
render_dir = f'{MODEL_PATH}/train/ours_30000/renders/'

if os.path.exists(gt_dir):
    gts = sorted(glob.glob(f'{gt_dir}*.png'))[:8]
    renders = sorted(glob.glob(f'{render_dir}*.png'))[:8]
    
    fig, axes = plt.subplots(2, 8, figsize=(24, 6))
    for i in range(min(8, len(gts))):
        axes[0, i].imshow(plt.imread(gts[i])); axes[0, i].axis('off')
        axes[1, i].imshow(plt.imread(renders[i])); axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Ground Truth', fontsize=11, rotation=0, labelpad=70, va='center')
    axes[1, 0].set_ylabel('Gaussian Splat', fontsize=11, rotation=0, labelpad=70, va='center')
    plt.suptitle('Ground Truth vs 3D Gaussian Splatting (30k iter, v4)', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/gs_gt_vs_render_v4.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('GT directory not found')

In [ ]:
# Metrics summary
print('=== TRAINING METRICS ===')
for iteration in [7000, 15000, 30000]:
    log_line = !grep -A2 "ITER {iteration}" /content/output/log.txt 2>/dev/null || echo ''
    if log_line:
        print(f'  {iteration} iter: {" ".join(log_line)}')

# Model size comparison
print('\n=== MODEL SIZE ===')
for p in sorted(glob.glob(f'{MODEL_PATH}/point_cloud/*/point_cloud.ply')):
    iter_name = os.path.basename(os.path.dirname(p))
    print(f'  {iter_name}: {os.path.getsize(p)/1e6:.1f} MB')

In [ ]:
# Save to Drive
SAVE_DIR = f'{DRIVE}/output_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save visualizations
for f in glob.glob('/content/gs_*.png'):
    shutil.copy(f, SAVE_DIR)
    print(f'Saved: {os.path.basename(f)}')

# Save final model
final_ply = f'{MODEL_PATH}/point_cloud/iteration_30000/point_cloud.ply'
if os.path.exists(final_ply):
    shutil.copy(final_ply, f'{SAVE_DIR}/point_cloud_30k.ply')
    print(f'Saved model: {os.path.getsize(final_ply)/1e6:.1f} MB')

# Save individual renders (best quality)
renders = sorted(glob.glob(f'{MODEL_PATH}/train/ours_30000/renders/*.png'))[:10]
for r in renders:
    shutil.copy(r, f'{SAVE_DIR}/{os.path.basename(r)}')
print(f'Saved {len(renders)} renders')

print(f'\n\u2705 All saved to {SAVE_DIR}')

## Summary

| Step | Details | Result |
|------|---------|--------|
| Extraction | 432 images (36 pos \u00d7 6 yaw \u00d7 2 pitch), flipped right-side-up | Ground-facing |
| COLMAP | OPENCV camera model, local CPU (33 min) | **432/432 (100%)**, 240k points |
| Training | 30k iter, densify\u219215k, SH=3, A100 | Crisp 3D model |

**Improvements over v3:**
- 100% image registration (was 89%)
- 4.7\u00d7 more 3D points (240k vs 51k)
- OPENCV camera model handles residual distortion
- Correct image orientation eliminates geometric confusion

**If still blurry:** The input images from equirectangular extraction have inherent softness. For sharper results, use the raw LRF dual-fisheye with direct fisheye-aware Gaussian Splatting (360-GS).